# Notebook 1: Build a graph "backbone" from structured data sources

Import the Python library dependencies.

In [1]:
import json
import pathlib

from sz_semantics import SzClient, Thesaurus
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-20T10:07:17.875342+00:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

json        : 2.0.9
sz_semantics: 1.3.3
watermark   : 2.5.0



## Open Data

In this tutorial we will construct and analyze an _investigative graph_, connecting "risk" data and "link" data within a graph. From this we can show patterns of criminal _tradecraft_ used in money laundering, tax evasion, and so on.

We have selected "slices" of open data from two providers, where elements connect to produce interesting subgraphs about fraud networks:

  - <https://www.opensanctions.org/>
  - <https://www.openownership.org/>

### OpenSanctions

[OpenSanctions](https://www.opensanctions.org/) provides the "risk" category of data.
In other words, this describes people and organizations who are known risks for FinCrime.
There is also the [`yente`](https://github.com/opensanctions/yente) API which provides
HTTP endpoints based on the [_FollowTheMoney_](https://followthemoney.tech/) data model
used for investigations and [OSInt](https://osintframework.com/).

### Open Ownership

[Open Ownership](https://www.openownership.org/) provides the "link" category of data.
This describes [_ultimate beneficial ownership_](https://www.beneficialownership.co.uk/)
(UBO) details: "Who owns how much of what, and who actually has controlling interest?"
There's also the [_Beneficial Ownership Data Standard_](https://standard.openownership.org/en/0.4.0/)
(BODS) which is an open standard providing guidance for collecting, sharing, and using
high-quality beneficial ownership data, to support corporate ownership transparency.

Open Ownership has partnered with [GLEIF](https://www.gleif.org/) to
launch the [_Global Open Data Integration Network_](https://godin.gleif.org)
(GODIN) to promote open standards across the world for data interoperability
among these kinds of datasets related to investigating transnational corruption.

Note: there is also a repository which publishes these datasets, already formatted as JSONL files for use in Senzing <https://www.opensanctions.org/docs/bulk/senzing/>

However, those full sources are a lot to download, so for this tutorial we're using selected "slices" which will produce interesting subgraphs.
To download these slices of the `OpenSanctions` and `Open Ownership` datasets:

In [3]:
!wget https://raw.githubusercontent.com/DerwenAI/strwythura/refs/heads/main/data/badshah/open-sanctions.json \
  -O data/open-sanctions.json

--2025-11-20 10:07:35--  https://raw.githubusercontent.com/DerwenAI/senzing_starter_kit/refs/heads/main/senzing_rootfs/data/open-sanctions.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47604 (46K) [text/plain]
Saving to: ‘data/open-sanctions.json’

data/open-sanctions 100%[===================>]  46.49K  --.-KB/s    in 0.04s   

2025-11-20 10:07:35 (1.28 MB/s) - ‘data/open-sanctions.json’ saved [47604/47604]



In [4]:
!wget https://raw.githubusercontent.com/DerwenAI/strwythura/refs/heads/main/data/badshah/open-ownership.json \
  -O data/open-ownership.json

--2025-11-20 10:07:37--  https://raw.githubusercontent.com/DerwenAI/senzing_starter_kit/refs/heads/main/senzing_rootfs/data/open-ownership.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 362382 (354K) [text/plain]
Saving to: ‘data/open-ownership.json’

data/open-ownership 100%[===================>] 353.89K   242KB/s    in 1.5s    

2025-11-20 10:07:42 (242 KB/s) - ‘data/open-ownership.json’ saved [362382/362382]



This should create the two JSONL files in the `data` subdirectory:
* `data/open-sanctions.json`
* `data/open-ownership.json`

In [5]:
!ls -lta data/*.json

-rw-r--r--  1 paco  staff  362382 Nov 20 10:07 data/open-ownership.json
-rw-r--r--  1 paco  staff   47604 Nov 20 10:07 data/open-sanctions.json


Let's examine the results, by JSON _pretty-printing_ the first line in each file.

In [6]:
!head -1 data/open-sanctions.json | python3 -m json.tool

{
    "DATA_SOURCE": "OPEN-SANCTIONS",
    "RECORD_ID": "NK-25vyVFzt8vdJGgAXMRTwTJ",
    "RECORD_TYPE": "PERSON",
    "LAST_CHANGE": "2024-07-30T16:41:14",
    "NAMES": [
        {
            "NAME_TYPE": "PRIMARY",
            "NAME_FULL": "Abassin BADSHAH"
        }
    ],
    "RISKS": [
        {
            "TOPIC": "corp.disqual"
        }
    ],
    "ADDRESSES": [
        {
            "ADDR_FULL": "31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"
        }
    ],
    "DATES": [
        {
            "DATE_OF_BIRTH": "1985-05-12"
        }
    ],
    "COUNTRIES": [
        {
            "NATIONALITY": "gb"
        }
    ],
    "SOURCE_LINKS": [
        {
            "SOURCE_URL": "https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk"
        }
    ],
    "RELATIONSHIPS": [
        {
            "REL_POINTER_ROLE": "Directorship",
            "REL_POINTER_DOMAIN": "OPEN-SANCTIONS",
            "REL_POINTER

Note the _risk_ elements in this data, which are `"TOPIC"` files within the `"RISKS"` category.

Then take a look at the _link_ elements:

In [7]:
!head -1 data/open-ownership.json | python3 -m json.tool

{
    "DATA_SOURCE": "OPEN-OWNERSHIP",
    "RECORD_ID": "10094521532396971848",
    "statementDate": "2023-06-18",
    "RECORD_TYPE": "ORGANIZATION",
    "NAMES": [
        {
            "PRIMARY_NAME_ORG": "GOLD WYNN UK HOLDINGS LIMITED"
        }
    ],
    "REGISTRATION_DATE": "2020-03-18",
    "REGISTRATION_COUNTRY": "GB",
    "ADDRESSES": [
        {
            "ADDR_TYPE": "BUSINESS",
            "ADDR_FULL": "C/O Fladgate Llp, 16 Great Queen Street, London, WC2B 5DG",
            "ADDR_COUNTRY": "GB"
        }
    ],
    "IDENTIFIERS": [
        {
            "NATIONAL_ID_NUMBER": "12524623",
            "NATIONAL_ID_TYPE": "GB-COH",
            "NATIONAL_ID_COUNTRY": "GBR"
        }
    ],
    "LINKS": [
        {
            "OpenCorporates": "https://opencorporates.com/companies/gb/12524623"
        },
        {
            "OpenOwnership Register": "https://register.openownership.org/entities/18432059995972240708"
        }
    ],
    "RELATIONSHIPS": [
        {
          

## Run entity resolution

Make sure you have already launched the [`serve-grpc` container](https://hub.docker.com/r/senzing/serve-grpc) and have it running in the background, by executing the following command line in another terminal window:

```bash
docker run -it --publish 8261:8261 --rm senzing/serve-grpc
```

In [8]:
!docker ps

CONTAINER ID   IMAGE                COMMAND             CREATED         STATUS                   PORTS                                         NAMES
40b1b59537e1   senzing/serve-grpc   "/app/serve-grpc"   4 minutes ago   Up 4 minutes (healthy)   0.0.0.0:8261->8261/tcp, [::]:8261->8261/tcp   clever_kapitsa


Next we'll use [`sz_semantics`](https://github.com/senzing-garage/sz-semantics/) library to call the Senzing SDK via the [gRPC](https://grpc.io/) server running in that Docker container.

To get started on _entity resolution_, first we need to specify a namespace for the [data sources](https://senzing.zendesk.com/hc/en-us/articles/115002897308-Data-Source-Records-DSRs-Explained) to use in Senzing.

In [9]:
data_sources: dict[ str, str ] = {
    "OPEN-SANCTIONS": "data/open-sanctions.json",
    "OPEN-OWNERSHIP": "data/open-ownership.json",
}

We need to configure how to reach the the Senzing SDK which is running as a [_microservice_](https://aws.amazon.com/microservices/).
See the [gRPC server](https://github.com/senzing-garage/serve-grpc) documentation for more details.

In [10]:
config: dict[ str, dict ] = {
    "sz": { "grpc_server": "localhost:8261" }
}

Now we have the two parts needed to configure the Senzing SDK.

In [11]:
sz: SzClient = SzClient(config, data_sources)

Then in one line of Python, run [entity resolution](https://senzing.com/what-is-entity-resolution/) on the named datasets.

In [12]:
ents_batch: dict = sz.entity_resolution(data_sources)

Let's examine the JSON returned from a Senzing SDK ["GET_ENTITY"](https://senzing.com/docs/tutorials/get_entity_response/) call on the first resolved entity.

In [13]:
one_get: str = sz.get_entity(1)
json.loads(one_get)

{'RESOLVED_ENTITY': {'ENTITY_ID': 1,
  'ENTITY_NAME': 'Abassin Badshah',
  'FEATURES': {'ADDRESS': [{'FEAT_DESC': '31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL',
     'LIB_FEAT_ID': 3,
     'FEAT_DESC_VALUES': [{'FEAT_DESC': '31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL',
       'LIB_FEAT_ID': 3}]},
    {'FEAT_DESC': '3, Market Parade, 41 East Street, Bromley, BR1 1QN',
     'LIB_FEAT_ID': 5885,
     'USAGE_TYPE': 'PRIMARY',
     'FEAT_DESC_VALUES': [{'FEAT_DESC': '3, Market Parade, 41 East Street, Bromley, BR1 1QN',
       'LIB_FEAT_ID': 5885}]},
    {'FEAT_DESC': '31, Quernmore Close, Bromley, BR1 4EL',
     'LIB_FEAT_ID': 5499,
     'USAGE_TYPE': 'PRIMARY',
     'FEAT_DESC_VALUES': [{'FEAT_DESC': '31, Quernmore Close, Bromley, BR1 4EL',
       'LIB_FEAT_ID': 5499}]}],
   'DOB': [{'FEAT_DESC': '1985-05-01',
     'LIB_FEAT_ID': 5498,
     'FEAT_DESC_VALUES': [{'FEAT_DESC': '1985-05-01', 'LIB_FEAT_ID': 5498}]},
    {'FEAT_DESC': '1985-05-12',
     'LIB_FEAT_

Next, we will represent each entity in RDF.

## Generate a domain-specific thesaurus

As a next step toward generating graph "building blocks" in RDF, initialize a `Thesaurus` instance and load the [Senzing taxonomy](https://github.com/senzing-garage/sz-semantics/blob/main/domain.ttl) into it.

In [14]:
thesaurus: Thesaurus = Thesaurus()
thesaurus.load_source(Thesaurus.DOMAIN_TTL)

Let's examine how one JSON response from the Senzing SDK gets represented in RDF:

In [15]:
for rdf_frag in thesaurus.parse_iter(one_get, language = "en"):
    print(rdf_frag)


sz:1 rdf:type sz:Person ;
 skos:prefLabel "Abassin Badshah"@en ;
.
[] rdf:subject sz:1 ;
 rdf:predicate skos:exactMatch ;
 rdf:object sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ ;
 sz:match_key "INITIAL" ;
 sz:match_level "INITIAL" ;
.
sz:1 prov:wasDerivedFrom sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ .
sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ rdf:type sz:DataRecord ;
 dc:identifier "NK-25vyVFzt8vdJGgAXMRTwTJ" ;
 prov:wasQuotedFrom sz:ds_open-sanctions ;
.
sz:ds_open-sanctions rdf:type dcat:Dataset ;
 dc:identifier "open-sanctions" ;
.
[] rdf:subject sz:1 ;
 rdf:predicate skos:exactMatch ;
 rdf:object sz:ds_open-ownership_17207853441353212969 ;
 sz:match_key "+NAME+ADDRESS+NATIONALITY" ;
 sz:match_level "RESOLVED" ;
.
sz:1 prov:wasDerivedFrom sz:ds_open-ownership_17207853441353212969 .
sz:ds_open-ownership_17207853441353212969 rdf:type sz:DataRecord ;
 dc:identifier "17207853441353212969" ;
 prov:wasQuotedFrom sz:ds_open-ownership ;
.
sz:ds_open-ownership rdf:type dcat:D

Next we'll iterate through all the resolved entities:
1. generate RDF fragments from from the JSON response for "GET_ENTITY"
2. collect these RDF triples into a list
3. prepend the RDF namespace prefixes onto the load
4. load these triples into the RDF graph in the `Thesaurus` (internally using `RDFlib`)

In [16]:
for ent_json in sz.sz_engine.export_json_entity_report_iterator():
    for rdf_frag in thesaurus.parse_iter(ent_json, language = "en"):
        thesaurus.load_source_text(
            Thesaurus.RDF_PREAMBLE + rdf_frag,
            format = "turtle",
        )

How many triples have we just loaded?

In [17]:
len(thesaurus.rdf_graph)

6696

Now the RDF graph includes triples for both the Senzing taxonomy and the generated domain-specific thesaurus, so serialize these results into the `thesaurus.ttl` file. This is written in ["Turtle"](https://medium.com/wallscope/understanding-linked-data-formats-rdf-xml-vs-turtle-vs-n-triples-eb931dbe9827) format, which is arguably much simpler to read, automatically verified, and also more _composable_.

In [18]:
thesaurus_path: pathlib.Path = pathlib.Path("thesaurus.ttl")

thesaurus.save_source(
    thesaurus_path,
    format = "turtle",
)

Let's examine the triples in the top part of that file.

In [19]:
!head -200 thesaurus.ttl

@prefix adms: <http://www.w3.org/ns/adms#> .
@prefix bods: <https://vocab.openownership.org/terms#> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ftm: <https://schema.followthemoney.tech/#> .
@prefix nc: <http://release.niem.gov/niem/niem-core/5.0/#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix ppcl: <http://www.semantic-web.at/ppcl/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rad: <http://www.w3.org/ns/rad#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix sz: <https://github.com/senzing-garage/sz-semantics/wiki/ns#> .
@prefix void: <http://rdfs.org/ns/void#> .
@prefix wco: <https://id.oclc.org/worldcat/ontology/> .
@prefix wd: <http://www.wikidata.org/entity/> .
@prefix xsd: <http:

Now we have an RDF _semantic graph_ to use in the subsequent steps.
At this point, let's examine the [SKOS](https://www.w3.org/2004/02/skos/) taxonomy used by Senzing and how it integrates with other related vocabularies.
In another browser tab, open to <https://github.com/senzing-garage/sz-semantics/wiki/ns>

---